# Ball detection on labelled volleyball frames

Measures the ball detectors and the tracker of the project (`vball.ball_detect`, `vball.ball_track`) on the
**Volleyball dataset** (Ibrahim et al., CVPR 2016; 55 broadcast matches, clips of 41 frames at 1280 x 720) with
the ball annotated in every frame by Perez et al. (Pattern Recognition 2022). Official test split: 16 matches,
1,337 clips. Script: `scripts/eval_ball_detection.py`. For reference, the last cell runs WASB (Tarashima et al.,
BMVC 2023) with its published volleyball weights through its own evaluation on the same split.

Runs on **Kaggle** (GPU T4, Internet on) with the dataset
`sherif31/group-activity-recognition-volleyball` added as input (62 GB, read in place). Outputs in
`/kaggle/working/ball_eval_volleyball/`.

In [ ]:
import glob, os, subprocess, sys
WORK = '/tmp/work'; OUT = '/kaggle/working'
os.makedirs(WORK, exist_ok=True)
# the dataset is mounted under /kaggle/input/...; find the folder that holds the match folders 0..54
VIDEOS = next(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/videos/47', recursive=True))
print('videos:', VIDEOS, len(os.listdir(VIDEOS)), 'matches')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# code: this project (vball) and the upstream ball models it reads in place (external/, unmodified)
!git clone -q --depth 1 https://github.com/DL-Volleyball-Analysis/volleyball-analysis {WORK}/repo
!git clone -q --depth 1 https://github.com/asigatchov/fast-volleyball-tracking-inference {WORK}/repo/external/fast-volleyball-tracking-inference
!mkdir -p {WORK}/repo/models && cp {WORK}/repo/external/fast-volleyball-tracking-inference/models/VballNetV4c_seq9_grayscale_*.onnx {WORK}/repo/models/vballnet_v4c.onnx
!pip -q install -e {WORK}/repo --no-deps
!pip -q install onnxruntime-gpu gdown
import onnxruntime; print(onnxruntime.get_available_providers())

In [ ]:
# ball annotations (Perez et al.), 1.6 MB
ANN_ZIP = f'{WORK}/ball_ann.zip'
!gdown -q 1urZpZiiepC85JD1u3VeURgUpztRgI0yl -O {ANN_ZIP} && unzip -q -o {ANN_ZIP} -d {WORK}/ann
ANN = f'{WORK}/ann/volleyball_ball_annotation'
print(len(os.listdir(ANN)), 'matches annotated')

In [ ]:
!cd {WORK}/repo && python scripts/eval_ball_detection.py --videos {VIDEOS} --ann {ANN} --out {OUT}/ball_eval_volleyball

## Reference: WASB with its published volleyball weights (its own evaluation, tau = 4 px)
Not part of our pipeline; numbers are comparable in split and threshold, but WASB was trained on this
dataset's training split while the detectors above were not.

In [ ]:
try:
    W = f'{WORK}/wasb'
    !git clone -q --depth 1 https://github.com/nttcom/WASB-SBDT {W}
    !pip -q install hydra-core omegaconf
    os.makedirs(f'{W}/pretrained_weights', exist_ok=True)
    !gdown -q 1M9y4wPJqLc0K-z-Bo5DP8Ft5XwJuLqIS -O {W}/pretrained_weights/wasb_volleyball_best.pth.tar
    os.makedirs(f'{W}/datasets/volleyball', exist_ok=True)
    for name, src in (('videos', VIDEOS), ('volleyball_ball_annotation', ANN)):
        dst = f'{W}/datasets/volleyball/{name}'
        if not os.path.exists(dst): os.symlink(src, dst)
    !cd {W}/src && python main.py --config-name=eval dataset=volleyball model=wasb detector.model_path=../pretrained_weights/wasb_volleyball_best.pth.tar 2>&1 | tee {OUT}/wasb_volleyball_eval.txt | tail -30
except Exception as e:
    print('WASB reference run failed:', e)